# CKA Circuit Comparison (IID vs Non-IID)

Run FedMI to train and discover circuits on IID and Non-IID partitions, and use **Centered Kernel Alignment (CKA)** to compare how these models and circuits encode classes.

---

## 1 · Setup environment

In [ ]:
import os
REPO_URL = "https://github.com/ha405/FedMI.git"
BRANCH = "cvpr"

if not os.path.isdir("FedMI"):
    !git clone -b {BRANCH} {REPO_URL}
else:
    !cd FedMI && git pull origin {BRANCH}

os.chdir("FedMI")

from fedmi.env import setup, patch_config, print_info
setup()
print_info()

## 2 · Train IID Runtime

In [ ]:
from fedmi.config import ExperimentConfig
from fedmi.runner import ExperimentRunner

# IID Configuration
cfg_iid = ExperimentConfig()
cfg_iid.dataset_name      = "MNIST"
cfg_iid.num_clients        = 2
cfg_iid.num_rounds         = 3  # keep small for fast demo
cfg_iid.local_epochs       = 2
cfg_iid.partition_method   = "iid"
cfg_iid.train_mode         = "sparse"
cfg_iid.discovery_steps    = 50

patch_config(cfg_iid, output_subdir="exp_iid")

print("Training IID models and discovering circuits...")
runner_iid = ExperimentRunner(cfg_iid)
runner_iid.setup()
runner_iid.run()

## 3 · Train Non-IID Runtime

In [ ]:
# Non-IID Configuration (Dirichlet skew)
cfg_noniid = ExperimentConfig()
cfg_noniid.dataset_name      = "MNIST"
cfg_noniid.num_clients        = 2
cfg_noniid.num_rounds         = 3
cfg_noniid.local_epochs       = 2
cfg_noniid.partition_method   = "dirichlet"
cfg_noniid.dirichlet_alpha    = 0.1  # High skew
cfg_noniid.train_mode         = "sparse"
cfg_noniid.discovery_steps    = 50

patch_config(cfg_noniid, output_subdir="exp_noniid")

print("Training Non-IID models and discovering circuits...")
runner_noniid = ExperimentRunner(cfg_noniid)
runner_noniid.setup()
runner_noniid.run()

## 4 · CKA Comparison: Pre-head Latents

Compare the full-model representations (before the classification head) between the IID and Non-IID experiments. Do they encode the inputs securely or in visually different latent spaces?

In [ ]:
from fedmi.playground import CKACompareExperiment

class Args:
    pass
args_prehead = Args()
args_prehead.exp_a = cfg_iid.output_dir
args_prehead.exp_b = cfg_noniid.output_dir
args_prehead.client_a = 0
args_prehead.client_b = 0
args_prehead.mode = "prehead"
args_prehead.round = None
args_prehead.max_samples = 2000
args_prehead.output = None  # Heatmap skipped for single scalar

cka_exp_prehead = CKACompareExperiment(args_prehead)
cka_exp_prehead.run()

## 5 · CKA Comparison: Cross-Experiment Circuit Matching

Compare the activated circuit for matching classes across the IID and Non-IID client. A low score implies the networks build structurally or functionally isolated pathways for the same class.

In [ ]:
args_circuit = Args()
args_circuit.exp_a = cfg_iid.output_dir
args_circuit.exp_b = cfg_noniid.output_dir
args_circuit.client_a = 0    # First client from IID
args_circuit.client_b = 0    # First client from Non-IID
args_circuit.mode = "circuit"
args_circuit.source = "local"
args_circuit.round = None
args_circuit.round_key = "last"
args_circuit.classes = None  # Compute all overlapping classes automatically
args_circuit.layer = None    # Default: last conv/linear before head
args_circuit.max_samples = 2000
args_circuit.output = "cka_heatmap.png"

cka_exp_circuit = CKACompareExperiment(args_circuit)
cka_exp_circuit.run()

## 6 · View Heatmap

In [ ]:
from IPython.display import Image, display
if os.path.exists("cka_heatmap.png"):
    display(Image(filename="cka_heatmap.png"))
else:
    print("Heatmap not generated. Check if there were any overlapping classes.")